### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("./data")

Found 1 PDF files to process

Processing: Cover Letter - Zensar Technologies.pdf
  ✓ Loaded 1 pages

Total documents loaded: 1


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m146 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Cover Letter - Zensar Technologies', 'source': 'data/Cover Letter - Zensar Technologies.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Cover Letter - Zensar Technologies.pdf', 'file_type': 'pdf'}, page_content='Dear  Hiring  Team  at  Zensar  Technologies,  \nI  am  writing  to  express  my  interest  in  AI/ML  engineering  opportunities  at  Zensar  Technologies.  \nWith\n \nhands-on\n \nexperience\n \nbuilding\n \nbackend-driven\n \nGenerative\n \nAI\n \nsystems\n \nand\n \nproduction-oriented\n \nmachine\n \nlearning\n \npipelines,\n \nI\n \nbring\n \na\n \nstrong\n \nblend\n \nof\n \napplied\n \nresearch\n \nthinking\n \nand\n \nsystem-level\n \nengineering\n \nexecution.\n \nIn  my  recent  role  as  a  Graduate  Engineer  Trainee  at  Neilsoft,  I  worked  extensively  on  deep  \nlearning–based\n \ncomputer\n \nvision\n \nsystems\n \nusi

In [4]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [5]:
chunks=split_documents(all_pdf_documents)
chunks

Split 1 documents into 4 chunks

Example chunk:
Content: Dear  Hiring  Team  at  Zensar  Technologies,  
I  am  writing  to  express  my  interest  in  AI/ML  engineering  opportunities  at  Zensar  Technologies.  
With
 
hands-on
 
experience
 
building
 
...
Metadata: {'producer': 'Skia/PDF m146 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Cover Letter - Zensar Technologies', 'source': 'data/Cover Letter - Zensar Technologies.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Cover Letter - Zensar Technologies.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m146 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Cover Letter - Zensar Technologies', 'source': 'data/Cover Letter - Zensar Technologies.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Cover Letter - Zensar Technologies.pdf', 'file_type': 'pdf'}, page_content='Dear  Hiring  Team  at  Zensar  Technologies,  \nI  am  writing  to  express  my  interest  in  AI/ML  engineering  opportunities  at  Zensar  Technologies.  \nWith\n \nhands-on\n \nexperience\n \nbuilding\n \nbackend-driven\n \nGenerative\n \nAI\n \nsystems\n \nand\n \nproduction-oriented\n \nmachine\n \nlearning\n \npipelines,\n \nI\n \nbring\n \na\n \nstrong\n \nblend\n \nof\n \napplied\n \nresearch\n \nthinking\n \nand\n \nsystem-level\n \nengineering\n \nexecution.\n \nIn  my  recent  role  as  a  Graduate  Engineer  Trainee  at  Neilsoft,  I  worked  extensively  on  deep  \nlearning–based\n \ncomputer\n \nvision\n \nsystems\n \nusi

### embedding And vectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
import os

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [8]:
class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents", vector_size: int = 384):
        self.collection_name = collection_name
        # Note: If using local Qdrant, ensure it's running or use ":memory:" for testing
        self.client = QdrantClient(host="localhost", port=6333)
        self.vector_size = vector_size
        self._initialize_store()

    def _initialize_store(self):
        try:
            if not self.client.collection_exists(collection_name=self.collection_name):
                # Qdrant REQUIRES VectorParams (size and distance) during creation
                self.client.create_collection(
                    collection_name=self.collection_name,
                    vectors_config=VectorParams(size=self.vector_size, distance=Distance.COSINE),
                )
                print(f"Collection {self.collection_name} created.")
            else:
                print(f"Collection {self.collection_name} already exists.")
        except Exception as e:
            print(f"Error initializing Qdrant: {e}")
            raise

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        points = []
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Qdrant uses PointStruct to wrap ID, Vector, and Payload (metadata + text)
            points.append(PointStruct(
                id=str(uuid.uuid4()), # Qdrant IDs must be UUIDs or integers
                vector=embedding.tolist(),
                payload={
                    "page_content": doc.page_content,
                    "metadata": doc.metadata,
                    "doc_index": i
                }
            ))
        
        try:
            # Use self.client.upsert, NOT self.collection.upsert
            self.client.upsert(
                collection_name=self.collection_name,
                points=points
            )
            
            count_result = self.client.count(collection_name=self.collection_name)
            print(f"Total documents now in collection: {count_result.count}")
            
        except Exception as e:
            print(f"Error adding to Qdrant: {e}")
            raise
vectorstore=VectorStore()
vectorstore


Collection pdf_documents created.


In [9]:
chunks

[Document(metadata={'producer': 'Skia/PDF m146 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Cover Letter - Zensar Technologies', 'source': 'data/Cover Letter - Zensar Technologies.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Cover Letter - Zensar Technologies.pdf', 'file_type': 'pdf'}, page_content='Dear  Hiring  Team  at  Zensar  Technologies,  \nI  am  writing  to  express  my  interest  in  AI/ML  engineering  opportunities  at  Zensar  Technologies.  \nWith\n \nhands-on\n \nexperience\n \nbuilding\n \nbackend-driven\n \nGenerative\n \nAI\n \nsystems\n \nand\n \nproduction-oriented\n \nmachine\n \nlearning\n \npipelines,\n \nI\n \nbring\n \na\n \nstrong\n \nblend\n \nof\n \napplied\n \nresearch\n \nthinking\n \nand\n \nsystem-level\n \nengineering\n \nexecution.\n \nIn  my  recent  role  as  a  Graduate  Engineer  Trainee  at  Neilsoft,  I  worked  extensively  on  deep  \nlearning–based\n \ncomputer\n \nvision\n \nsystems\n \nusi

In [10]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 4 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (4, 384)
Total documents now in collection: 4


### Retriever Pipeline From VectorStore

In [26]:
class RAGRetriever:
    """Handles query-based retrieval from the Qdrant vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        try:
            # Use query_points instead of search
            response = self.vector_store.client.query_points(
                collection_name=self.vector_store.collection_name,
                query=query_embedding.tolist(),
                limit=top_k,
                score_threshold=score_threshold # Qdrant can filter by score natively!
            )

            retrieved_docs = []
            # Note: query_points returns an object with a .points attribute
            for i, result in enumerate(response.points):
                payload = result.payload
                retrieved_docs.append({
                    'id': result.id,
                    'content': payload.get('page_content', ''), # Matches the key from our previous 'upsert'
                    'metadata': payload.get('metadata', {}),
                    'similarity_score': result.score,
                    'rank': i + 1
                })

            return retrieved_docs            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

In [27]:
rag_retriever = RAGRetriever(vectorstore,embedding_manager)

In [28]:
rag_retriever.retrieve("What is attention is all you need")

Retrieving documents for query: 'What is attention is all you need'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)


[{'id': '89cd0256-9979-4747-b4bc-406a7918af18',
  'content': 'analysis\n \nto\n \nresolve\n \nclass\n \nimbalance\n \nand\n \nprediction\n \ninconsistencies.\n \nI\n \nalso\n \ndeveloped\n \na\n \nmulti-model\n \ninference\n \npipeline\n \nthat\n \nautomated\n \nbatch\n \npredictions,\n \npost-processing,\n \nand\n \nstructured\n \noutputs\n \n—\n \nfocusing\n \non\n \nreliability\n \nand\n \nproduction\n \nreadiness\n \nrather\n \nthan\n \nisolated\n \nmodel\n \nperformance.\n \nAlongside  computer  vision,  I  have  been  building  Generative  AI  systems  with  a  backend  \nengineering\n \nfocus.\n \nI\n \ndeveloped\n \na\n \nRAG-powered\n \nPDF\n \nIntelligence\n \nsystem\n \nthat\n \nintegrates\n \nembeddings,\n \nvector\n \nsearch,\n \nand\n \nLLM\n \nAPIs\n \nto\n \ndeliver\n \ngrounded,\n \ncontext-aware\n \nanswers\n \nwhile\n \nreducing\n \nhallucinations\n \nthrough\n \nprompt\n \nand\n \nchunking\n \noptimization.\n \nI\n \nalso\n \nbuilt\n \nan\n \nAgentic\n \nResearch\n 

In [29]:
rag_retriever.retrieve("Unified Multi-task Learning Framework")


Retrieving documents for query: 'Unified Multi-task Learning Framework'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)


[{'id': '89cd0256-9979-4747-b4bc-406a7918af18',
  'content': 'analysis\n \nto\n \nresolve\n \nclass\n \nimbalance\n \nand\n \nprediction\n \ninconsistencies.\n \nI\n \nalso\n \ndeveloped\n \na\n \nmulti-model\n \ninference\n \npipeline\n \nthat\n \nautomated\n \nbatch\n \npredictions,\n \npost-processing,\n \nand\n \nstructured\n \noutputs\n \n—\n \nfocusing\n \non\n \nreliability\n \nand\n \nproduction\n \nreadiness\n \nrather\n \nthan\n \nisolated\n \nmodel\n \nperformance.\n \nAlongside  computer  vision,  I  have  been  building  Generative  AI  systems  with  a  backend  \nengineering\n \nfocus.\n \nI\n \ndeveloped\n \na\n \nRAG-powered\n \nPDF\n \nIntelligence\n \nsystem\n \nthat\n \nintegrates\n \nembeddings,\n \nvector\n \nsearch,\n \nand\n \nLLM\n \nAPIs\n \nto\n \ndeliver\n \ngrounded,\n \ncontext-aware\n \nanswers\n \nwhile\n \nreducing\n \nhallucinations\n \nthrough\n \nprompt\n \nand\n \nchunking\n \noptimization.\n \nI\n \nalso\n \nbuilt\n \nan\n \nAgentic\n \nResearch\n 

### RAG Pipeline- VectorDB To LLM Output Generation

In [37]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [38]:
class GroqLLM:
    def __init__(self, model_name: str = "gemma2-9b-it", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: AI model name (qwen2-72b-instruct, llama3-70b-8192, etc.)

        """
        self.model_name = model_name
        
        self.llm = ChatOllama(
            model=self.model_name,
            temperature=0.1,
            num_predict=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    


In [ ]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("Summarize the document")

Retrieving documents for query: 'Summarize the document'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)


[{'id': '89cd0256-9979-4747-b4bc-406a7918af18',
  'content': 'analysis\n \nto\n \nresolve\n \nclass\n \nimbalance\n \nand\n \nprediction\n \ninconsistencies.\n \nI\n \nalso\n \ndeveloped\n \na\n \nmulti-model\n \ninference\n \npipeline\n \nthat\n \nautomated\n \nbatch\n \npredictions,\n \npost-processing,\n \nand\n \nstructured\n \noutputs\n \n—\n \nfocusing\n \non\n \nreliability\n \nand\n \nproduction\n \nreadiness\n \nrather\n \nthan\n \nisolated\n \nmodel\n \nperformance.\n \nAlongside  computer  vision,  I  have  been  building  Generative  AI  systems  with  a  backend  \nengineering\n \nfocus.\n \nI\n \ndeveloped\n \na\n \nRAG-powered\n \nPDF\n \nIntelligence\n \nsystem\n \nthat\n \nintegrates\n \nembeddings,\n \nvector\n \nsearch,\n \nand\n \nLLM\n \nAPIs\n \nto\n \ndeliver\n \ngrounded,\n \ncontext-aware\n \nanswers\n \nwhile\n \nreducing\n \nhallucinations\n \nthrough\n \nprompt\n \nand\n \nchunking\n \noptimization.\n \nI\n \nalso\n \nbuilt\n \nan\n \nAgentic\n \nResearch\n 

### Integration Vectordb Context pipeline With LLM output

In [47]:
### Simple RAG pipeline with Groq LLM
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)

llm=ChatOllama(model="phi3",temperature=0.1,num_predict=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [49]:
answer=rag_simple("Summarize the document",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'Summarize the document'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Aditya has developed a multi-model inference pipeline using RAG and FastAPI for AI services with asynchronous Python capabilities, focusing on reliability in real-world environments rather than isolated model performance. He also built an Agentic Research Assistant API that employs LLM agents to perform complex tasks like tool orchestration through memory-based context handling via asynchrony pipelines. Aditya's expertise lies in designing RAG pipelines, building AI services with asynchronous Python, integrating LLM SDKs for structured outputs and workflows, developing scalable ML systems prioritizing reproducibility, and he is particularly motivated by roles intersecting AI, backend engineering, and system design. He expresses interest in contributing to Zensar's AI initiatives where his background can support the team’s goals.


### Enhanced RAG Pipeline Features

In [44]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Hard Negative Mining Technqiues", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Hard Negative Mining Technqiues'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Answer: Hard negative mining techniques involve the strategic selection and incorporation of challenging, misclassified examples into training data to improve model robustness. These methods often use active learning or uncertainty sampling where models identify samples that they are most uncertain about for manual annotation and subsequent retraining. This process helps in refining classifiers by focusing on difficult cases which the AI struggles with, thereby enhancing its ability to generalize from training data to real-world scenarios.
Sources: [{'source': 'Cover Letter - Zensar Technologies.pdf', 'page': 0, 'score': 0.1366246, 'preview': 'Dear  Hiring  Team  at  Zensar  Technologies,  \nI  am  writing  to  express  my  interest  in  AI/ML  engineering  opportunities  at  Zensar  Technologies.  \nWith\n \nhands-on\n \nexperience\n \nbuilding\n \nbackend-driven\n \nGenerative\n \nAI\n \nsystems\n \nand\n \nproduction-oriented\n \nmachine\n \

In [45]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is attention is all you need'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Streaming answer:
Use the following context to answer the question concisely.
Context:
analysis
 
to
 
resolve
 
class
 
imbalance
 
and
 
prediction
 
inconsistencies.
 
I
 
also
 
developed
 
a
 
multi-model
 
inference
 
pipeline
 
that
 
automated
 
batch
 
predictions,
 
post-processing,
 
and
 
structured
 
outputs
 
—
 
focusing
 
on
 
reliability
 
and
 
production
 
readiness
 
rather
 
than
 
isolated
 
model
 
performance.
 
Alongside  computer  vision,  I  have  been  building  Generative  AI  systems  with  a  backend  
engineering
 
focus.
 
I
 
developed
 
a
 
RAG-powered
 
PDF
 
Intelligence
 
system
 
that
 
integrates
 
embeddings,
 
vector
 
search,
 
and
 
LLM
 
APIs
 
to
 
deliver
 
grounded,
 
context-aware
 
answers
 
while
 
reducing
 
hallucinations
 
through
 
prompt
 
and
 
chunking
 
optimization.
 
I
 
also
 
built
 
an
 
Agentic
 
Research
 
Assistant
 
API
 
using
 
FastAPI,
 
where
 
LLM-driven
 
agents
 
perform